In [3]:
def evaluate_season(season):
    """Refit the whole pipeline on one season and report test-set scores."""
    from src.utils.pipeline import load_all_snapshots
    from src.models.whiff import build_features

    df = load_all_snapshots(seasons=[season])

    # Same month-day boundaries as the 2024 constants, year substituted.
    # The All-Star break moves a few days year to year; this keeps the
    # split chronological and roughly proportioned, which is what a
    # season-to-season comparison needs.
    wm = whiff.build(df,
                     train_end=f"{season}-07-14",
                     validation_end=f"{season}-08-31")
    gb = fit_boosting(wm.X_train, wm.y_train)

    test = wm.split.test
    X_test = build_features(test, wm.keep_types, columns=wm.X_train.columns)
    y = test["target"].to_numpy()

    p_gb = gb.predict_proba(X_test)[:, 1]
    p_base = wm.predict_baseline(test)

    return {
        "season": season,
        "train_swings": len(wm.X_train),
        "test_swings": len(test),
        "test_whiff": y.mean(),
        "baseline_ll": log_loss(y, p_base),
        "boosting_ll": log_loss(y, p_gb),
        "auc": roc_auc_score(y, p_gb),
    }

rows = [evaluate_season(s) for s in [2021, 2022, 2023, 2024]]
r = pd.DataFrame(rows)
r["improvement"] = (r["baseline_ll"] - r["boosting_ll"]) / r["baseline_ll"]
print(r.round(4).to_string(index=False))

 season  train_swings  test_swings  test_whiff  baseline_ll  boosting_ll    auc  improvement
   2021        184266        62620      0.2351       0.4945       0.4468 0.7682       0.0965
   2022        186045        66720      0.2398       0.4989       0.4541 0.7650       0.0899
   2023        190489        60037      0.2370       0.4955       0.4503 0.7662       0.0913
   2024        200735        54137      0.2413       0.4947       0.4469 0.7774       0.0965


In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.models.batter_score import (
    SKILL_METRICS, build_profiles, fit, qualified,
)

SEASONS = [2021, 2022, 2023, 2024]

fits, profs = {}, {}
for s in SEASONS:
    df = load_all_snapshots(seasons=[s])
    p = build_profiles(df)
    profs[s] = p
    fits[s] = fit(p)
    print(f"{s}: {len(qualified(p))} qualified batters")

w = pd.DataFrame({s: fits[s].weights for s in SEASONS}).T
w.index.name = "season"
print()
print("learned weights by season:")
print(w.round(4).to_string())
print()
print("coefficient of variation:")
print((w.std() / w.mean().abs()).round(3).to_string())
print()
print("R-squared:", {s: round(fits[s].r_squared, 3) for s in SEASONS})

2021: 327 qualified batters
2022: 334 qualified batters
2023: 343 qualified batters
2024: 343 qualified batters

learned weights by season:
        chase_pct  zone_swing_pct  zone_contact_pct  barrel_pct
season                                                         
2021      -0.0086          0.0075            0.0174      0.0307
2022      -0.0026          0.0030            0.0198      0.0287
2023      -0.0068          0.0034            0.0161      0.0283
2024      -0.0075          0.0067            0.0184      0.0289

coefficient of variation:
chase_pct           0.414
zone_swing_pct      0.438
zone_contact_pct    0.086
barrel_pct          0.037

R-squared: {2021: 0.483, 2022: 0.453, 2023: 0.474, 2024: 0.501}


In [5]:
from scipy.stats import spearmanr

# Score every season's players with every season's weights.
# If the weights are a property of baseball rather than of 2024, the
# ranking should barely move.
rows = []
for target in SEASONS:
    q = qualified(profs[target])
    for source in SEASONS:
        scores = fits[source].score(q)
        if source == target:
            own = scores
        rows.append({"players_from": target, "weights_from": source,
                     "scores": scores})

print("rank correlation: each season's players, scored by each season's weights")
print()
out = []
for target in SEASONS:
    q = qualified(profs[target])
    base = fits[target].score(q)
    line = {"players_from": target}
    for source in SEASONS:
        alt = fits[source].score(q)
        line[f"w{source}"] = round(spearmanr(base, alt).statistic, 4)
    out.append(line)
print(pd.DataFrame(out).to_string(index=False))

rank correlation: each season's players, scored by each season's weights

 players_from  w2021  w2022  w2023  w2024
         2021 1.0000 0.9729 0.9926 0.9983
         2022 0.9707 1.0000 0.9812 0.9809
         2023 0.9906 0.9770 1.0000 0.9944
         2024 0.9976 0.9760 0.9927 1.0000


In [6]:
# Different question: does a player keep his score year to year?
# This is about players changing, not weights changing.
common = None
score_by_season = {}
for s in SEASONS:
    q = qualified(profs[s])
    score_by_season[s] = fits[2024].score(q)   # one fixed set of weights
    common = set(q.index) if common is None else common & set(q.index)

common = sorted(common)
print(f"{len(common)} batters qualified in all four seasons")
print()

m = pd.DataFrame({s: score_by_season[s].reindex(common) for s in SEASONS})
print("year-to-year correlation of the score itself:")
print(m.corr().round(3).to_string())

145 batters qualified in all four seasons

year-to-year correlation of the score itself:
       2021   2022   2023   2024
2021  1.000  0.649  0.661  0.633
2022  0.649  1.000  0.748  0.720
2023  0.661  0.748  1.000  0.714
2024  0.633  0.720  0.714  1.000
